In [ ]:
# Configure ADLS access using your Databricks workspace identity or secret scope.
storage_account_name = "YOUR_STORAGE_ACCOUNT"


In [ ]:
reviews_path = f"abfss://processed@{storage_account_name}.dfs.core.windows.net/reviews/"
reviews_df = spark.read.parquet(reviews_path)


In [ ]:
reviews_df.printSchema()
display(reviews_df.limit(5))

In [ ]:
from pyspark.sql.functions import col, trim, length
clean_reviews_df = reviews_df
# Drop rows with missing critical fields
clean_reviews_df = clean_reviews_df.filter(
col("asin").isNotNull() &
col("reviewerID").isNotNull() &
col("overall").isNotNull()
)
# Enforce valid rating values
clean_reviews_df = clean_reviews_df.filter(
(col("overall") >= 1) & (col("overall") <= 5)
)
# Clean review text
clean_reviews_df = clean_reviews_df.withColumn(
"reviewText",
trim(col("reviewText"))
).filter(
length(col("reviewText")) >= 10
)

In [ ]:
print("Rows before cleaning:", reviews_df.count())
print("Rows after cleaning:", clean_reviews_df.count())
display(clean_reviews_df.limit(5))

In [ ]:
clean_reviews_path = f"abfss://processed@{storage_account_name}.dfs.core.windows.net/clean_reviews/"
clean_reviews_df.write.mode("overwrite").parquet(clean_reviews_path)
